# 02 · Controlled head-to-head comparison with re-implemented baselines

Reviewer 1 (comments 4 and 10) and Reviewer 2 (comment 4) asked for a comparison under a common protocol. Every model below is therefore trained and evaluated:
* on **identical partitions**: official train/test files, the same contiguous validation block, and the same chronological CICIDS-2018 subset;
* with **identical preprocessing** (one-hot + Min–Max fitted on the training split) and **class-weighted loss** (no resampling for any model);
* on **identical test targets** (the final events of the TA-LSTMAE test windows), with the same threshold (0.5) and the same metrics engine, including **causal MTTD for every model**;
* with the **same seeds** (`cfg.seeds`), reporting mean ± SD, 95% CIs and paired tests.

| Model | Role |
|---|---|
| TA-LSTMAE (TAE + HBLT, full) | proposed |
| TA-LSTMAE causal (TAE + unidirectional LSTM) | simplified, strictly causal candidate (R1-5) |
| TA-LSTMAE simplified (TAE + CNN only) | smallest variant (R1-5) |
| HAE-HRL (re-implemented) | strongest baseline in the original comparison scope |
| ADFCNN-BiLSTM (re-implemented) | representative recent (2025) attention / deformable-CNN / BiLSTM method (R1-10) |

Unreported baseline hyperparameters are listed in the next cell and must be stated in the manuscript. Run notebook 01 first only if you want the shared outputs folder; this notebook is self-contained.

In [ ]:
import os, sys, json, warnings
from pathlib import Path
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
ROOT = Path.cwd() if (Path.cwd() / "talstmae").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import tensorflow as tf
from talstmae import Config, set_all_seeds
from talstmae import data as DT, models as M, metrics as MT, experiment as X, alerts as AL

SMOKE_TEST = False      # True = tiny synthetic data, ONLY to check that the code executes. Never report its numbers.
RESUME = True           # continue an interrupted run (finished models/seeds are skipped)
cfg = Config().smoke() if SMOKE_TEST else Config()
if os.environ.get("TALSTMAE_SETTINGS"):          # settings written by 00_RUN_ME.ipynb override the lines above
    _s = json.loads(Path(os.environ["TALSTMAE_SETTINGS"]).read_text())
    _base = Config(**{k: (tuple(v) if k in ("datasets", "seeds") else v) for k, v in _s.get("config", {}).items()})
    SMOKE_TEST, RESUME = bool(_s.get("smoke_test")), bool(_s.get("resume", True))
    cfg = _base.smoke() if SMOKE_TEST else _base
    print("settings from 00_RUN_ME:", _s)
if SMOKE_TEST:
    print("=" * 78 + "\nSMOKE TEST MODE: synthetic placeholder data. Results are INVALID and must not be reported.\n" + "=" * 78)
set_all_seeds(cfg.seed, cfg.deterministic_ops)
print("TensorFlow", tf.__version__, "| outputs ->", cfg.out.resolve())

from talstmae import baselines as B
for k, v in B.BASELINE_NOTES.items():
    print(f"• {k}: {v}\n")

In [ ]:
DATA = {ds: DT.prepare_dataset(ds, cfg) for ds in cfg.datasets}

## 1. Train all models × seeds × datasets
Run time is roughly 5 models × 3 seeds × 3 datasets (the reported run uses seeds 42–44). On one modern GPU, expect several hours.

In [ ]:
RUNS = X.run_head_to_head(DATA, cfg, models=X.HEAD_TO_HEAD_MODELS, seeds=cfg.seeds, resume=RESUME)
RUNS.head()

## 2. Summary: mean ± SD and 95% confidence intervals

In [ ]:
SUM = MT.summarize_runs(RUNS)
SUM.to_csv(cfg.out / "head_to_head_summary.csv", index=False)
TAB = MT.format_mean_sd(SUM, ("accuracy", "precision", "recall", "f1", "MTTD_causal"))
TAB["params"] = RUNS.groupby(["dataset", "model"], sort=False)["params"].first().values
TAB["train_s_mean"] = RUNS.groupby(["dataset", "model"], sort=False)["train_seconds"].mean().round(0).values
TAB.to_csv(cfg.out / "tables_head_to_head.csv", index=False)
TAB

## 3. Significance
* **Across seeds:** a paired t-test (and Wilcoxon, when n ≥ 5) on F1 between the proposed model and every other model, with Holm adjustment.
* **Within seed 42:** an exact McNemar test on the paired test-set predictions.

In [ ]:
tests = []
for ds in DATA:
    for other in X.HEAD_TO_HEAD_MODELS[1:]:
        for metric in ("f1", "accuracy"):
            tests.append(MT.paired_seed_test(RUNS, metric, X.TA_FULL, other, ds))
TESTS = pd.DataFrame(tests)
ok = TESTS.paired_t_p.notna()
TESTS["paired_t_p_holm"] = np.nan
TESTS.loc[ok, "paired_t_p_holm"] = MT.holm(TESTS.loc[ok, "paired_t_p"].values)
mc = []
for ds in DATA:
    a = X.load_run_predictions(cfg, ds, X.TA_FULL, cfg.seeds[0])
    for other in X.HEAD_TO_HEAD_MODELS[1:]:
        b = X.load_run_predictions(cfg, ds, other, cfg.seeds[0])
        mc.append(dict(dataset=ds, model_b=other, mcnemar_p=MT.mcnemar_exact(
            a["y_true"], (a["prob_test"] >= cfg.threshold).astype(int), (b["prob_test"] >= cfg.threshold).astype(int))))
TESTS.to_csv(cfg.out / "head_to_head_paired_tests.csv", index=False)
pd.DataFrame(mc).to_csv(cfg.out / "head_to_head_mcnemar_seed42.csv", index=False)
display(TESTS.round(4)); display(pd.DataFrame(mc).round(4))

## 4. Reading the result (Table 7)
* Differences between the proposed model and the re-implemented baselines are tested with Holm-adjusted paired t-tests across seeds and with exact McNemar tests on the seed-42 predictions. In the reported run no difference is significant, so the manuscript reports **parity** and places the contribution in the operational-metrics layer (R1-10, R2-1).
* The causal-only and CNN-only variants are compared with the full model in the same way; TAE + CNN is recommended for deployment (R1-5).
* MTTD is measured for **every** model with the same engine and timestamps, so Table 7 reports causal MTTD for the re-implemented baselines as well.